# CSCE 40103 Module 3, Lecture 1 Code Notebook

## Rule-Based Email Risk Scoring

This notebook supports the first Module 3 lecture: **From Inbox to Attack Vector**.

The goal is not to build a real phishing system. The goal is to understand how suspicious clues can be converted into measurable features and how a simple rule-based detector makes decisions.

## Safety boundary

This notebook uses only fake classroom messages.

Do not use this notebook to send emails, create phishing pages, collect credentials, test real people, or impersonate real organizations. All links are defanged, for example `hxxp://example[.]com/login`, so they are safe to discuss in class.

## 1. Import packages

We use `pandas` only to display the message features in a clear table.

In [10]:
import pandas as pd

## 2. Create fake classroom messages

These examples are safe and fictional. Some are normal university-style messages. Some are suspicious-style messages for defensive analysis.

In [11]:
messages = [
    "Your Example University account will expire today. Review your access profile at hxxp://example[.]com/login.",
    "Reminder: your project notebook is due Friday on Blackboard.",
    "Congratulations, you won a free gift card. Claim now.",
    "Your scholarship form is due today. Use the official university portal.",
    "Your mailbox storage is full. Update your account settings immediately.",
    "Urgent: your assignment submission link closes tonight. Please log in to Blackboard."
]

for i, message in enumerate(messages, start=1):
    print(f"Message {i}: {message}")

Message 1: Your Example University account will expire today. Review your access profile at hxxp://example[.]com/login.
Message 2: Reminder: your project notebook is due Friday on Blackboard.
Message 3: Congratulations, you won a free gift card. Claim now.
Message 4: Your scholarship form is due today. Use the official university portal.
Message 5: Your mailbox storage is full. Update your account settings immediately.
Message 6: Urgent: your assignment submission link closes tonight. Please log in to Blackboard.


## 3. Define clue lists

A rule-based detector needs manually selected clues. These clues are not perfect. They are simple classroom examples.

In [12]:
urgent_words = ["urgent", "today", "immediately", "expire", "expires", "disabled", "now", "midnight"]
account_words = ["account", "password", "login", "access", "profile", "mailbox", "verify", "verification"]
money_words = ["gift", "prize", "refund", "payment", "scholarship", "cash", "reward"]

## 4. Extract simple features

A human sees suspicious clues. A model needs measurable features.

Examples:

- "This message has a link" becomes `has_link`
- "This message sounds urgent" becomes `urgent_word_count`
- "This message mentions account access" becomes `account_word_count`

In [13]:
def count_matches(text, word_list):
    """Count how many words from word_list appear in text."""
    text = text.lower()
    return sum(word in text for word in word_list)


def extract_features(message):
    """Convert one message into a small dictionary of measurable features."""
    text = message.lower()

    features = {
        "message_length": len(message),
        "has_link": int("hxxp" in text or "http" in text),
        "urgent_word_count": count_matches(text, urgent_words),
        "account_word_count": count_matches(text, account_words),
        "money_word_count": count_matches(text, money_words),
        "exclamation_count": message.count("!"),
    }

    return features

# Test the function on one message.
extract_features(messages[0])

{'message_length': 108,
 'has_link': 1,
 'urgent_word_count': 2,
 'account_word_count': 4,
 'money_word_count': 0,
 'exclamation_count': 0}

## 5. Convert all messages into a feature table

Each row is one message. Each column is one measurable feature.

In [14]:
feature_rows = []

for message in messages:
    row = extract_features(message)
    row["message"] = message
    feature_rows.append(row)

features_df = pd.DataFrame(feature_rows)
features_df = features_df[["message", "message_length", "has_link", "urgent_word_count", "account_word_count", "money_word_count", "exclamation_count"]]

display(features_df)

,message,message_length,has_link,urgent_word_count,account_word_count,money_word_count,exclamation_count
0,Your Example University account will expire to...,108,1,2,4,0,0
1,Reminder: your project notebook is due Friday ...,60,0,0,0,0,0
2,"Congratulations, you won a free gift card. Cla...",53,0,1,0,1,0
3,Your scholarship form is due today. Use the of...,71,0,1,0,1,0
4,Your mailbox storage is full. Update your acco...,71,0,1,2,0,0
5,Urgent: your assignment submission link closes...,84,0,1,0,0,0


## 6. Create a simple risk score

This is **not machine learning**. It is a manually written scoring rule.

The score uses feature weights that we choose by hand.

In [15]:
def calculate_risk_score(message):
    features = extract_features(message)

    score = 0
    score += features["has_link"] * 2
    score += features["urgent_word_count"] * 1
    score += features["account_word_count"] * 1
    score += features["money_word_count"] * 1
    score += min(features["exclamation_count"], 3)

    return score


def choose_action(score):
    """Map a risk score to a classroom action."""
    if score >= 5:
        return "Human review or quarantine"
    elif score >= 3:
        return "Warning banner"
    else:
        return "Inbox"

# Test on the first message.
score = calculate_risk_score(messages[0])
print("Score:", score)
print("Suggested action:", choose_action(score))

Score: 8
Suggested action: Human review or quarantine


## 7. Apply the scoring rule to all messages

Now we can see which messages are flagged by the simple rule.

In [16]:
results = []

for message in messages:
    score = calculate_risk_score(message)
    action = choose_action(score)
    results.append({"message": message, "risk_score": score, "suggested_action": action})

results_df = pd.DataFrame(results)
display(results_df)

,message,risk_score,suggested_action
0,Your Example University account will expire to...,8,Human review or quarantine
1,Reminder: your project notebook is due Friday ...,0,Inbox
2,"Congratulations, you won a free gift card. Cla...",2,Inbox
3,Your scholarship form is due today. Use the of...,2,Inbox
4,Your mailbox storage is full. Update your acco...,3,Warning banner
5,Urgent: your assignment submission link closes...,1,Inbox


## 8. Inspect possible false alarms

A rule may overreact to normal messages.

For example, a legitimate class message can contain words like `urgent`, `link`, `login`, and `submit`. That does not automatically make it malicious.

In [17]:
normal_but_suspicious_looking = "Urgent: your assignment submission link closes tonight. Please log in to Blackboard."

print(normal_but_suspicious_looking)
print("Features:", extract_features(normal_but_suspicious_looking))
print("Risk score:", calculate_risk_score(normal_but_suspicious_looking))
print("Suggested action:", choose_action(calculate_risk_score(normal_but_suspicious_looking)))

Urgent: your assignment submission link closes tonight. Please log in to Blackboard.
Features: {'message_length': 84, 'has_link': 0, 'urgent_word_count': 1, 'account_word_count': 0, 'money_word_count': 0, 'exclamation_count': 0}
Risk score: 1
Suggested action: Inbox


## 9. Add your own safe test message

Use fake names and defanged links only. Do not test real people.

Change the message below and rerun the cell.

In [18]:
my_test_message = "Your Example University account access will be disabled today. Review your account at hxxp://example[.]com/login."

print("Message:", my_test_message)
print("Features:", extract_features(my_test_message))
print("Risk score:", calculate_risk_score(my_test_message))
print("Suggested action:", choose_action(calculate_risk_score(my_test_message)))

Message: Your Example University account access will be disabled today. Review your account at hxxp://example[.]com/login.
Features: {'message_length': 113, 'has_link': 1, 'urgent_word_count': 2, 'account_word_count': 3, 'money_word_count': 0, 'exclamation_count': 0}
Risk score: 7
Suggested action: Human review or quarantine


## 10. What this rule-based detector teaches us

A rule-based detector is useful for learning because it is transparent. You can see every feature and every score.

But it is limited:

- It does not learn from data.
- It may block normal urgent messages.
- Attackers can change wording.
- It does not check the real sender.
- It does not inspect the true link destination.

In the next notebook, we move from hand-written rules to a model that learns from labeled examples.